# 🏥 Bupa RAG — 데이터 전처리 (Section-Aware Preprocessing)

## 전처리 전략 요약

### 포함 섹션 및 section_type 태그
| section_type | 해당 섹션 | 이유 |
|---|---|---|
| `benefit_table` | Table of Benefits | 보장 금액·한도 — RAG 핵심 |
| `exclusion` | General Exclusions / What is not covered | 미보장 항목 — 가장 많이 질문 |
| `claim_process` | The Claiming Process | 청구 방법·서류·기간 |
| `pre_auth` | Pre-authorisation / Need Treatment | 사전승인 필요 항목·절차 |
| `waiting_period` | Waiting Periods (표 안 텍스트) | 몇 달 후 보장되는지 |
| `glossary` | Glossary | 용어 정의 — 약관 해석에 필수 |
| `membership_admin` | Want to Add More People / Dependants | 피부양자·신생아 추가 절차 |
| `terms_conditions` | Terms & Conditions (운영 조항만) | 청구기한·해지·기존질환 등 |

### 제외 섹션
| 섹션 | 이유 |
|---|---|
| 커버/인트로 페이지 | 브랜딩, 정보 없음 |
| MembersWorld 앱 안내 | 앱 사용법, 보험 내용 없음 |
| Wellbeing Services | 홍보성 부가서비스 |
| Privacy Notice / GDPR | 개인정보 처리 방침 |
| 'Hello' 인트로 | 인사말, 정보 없음 |
| 연락처 마지막 페이지 | 전화번호만, 내용 없음 |

## 0. 패키지 설치

In [1]:
# !pip install pymupdf pdfplumber langchain langchain-community langchain-huggingface \
#              langchain-openai langchain-chroma langgraph sentence-transformers \
#              chromadb python-dotenv -q

In [2]:
# !pip install llama-parse python-dotenv -q

## 1. 환경 설정

In [3]:
import os, re
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()

# 나머지 코드 그대로...
load_dotenv()
print(f"OpenAI API Key: {'✅' if os.getenv('OPENAI_API_KEY') else '❌'}")
print(f"LLAMA CLOUD API Key: {'✅' if os.getenv('LLAMA_CLOUD_API_KEY') else '❌'}")

OpenAI API Key: ✅
LLAMA CLOUD API Key: ✅


## 2. PDF 설정 (Business Plan 제외)

In [4]:
PDF_BASE_DIR = './data'

PDF_CONFIGS = [
    {
        'path': os.path.join(PDF_BASE_DIR, 'HKX_IHHP_Membership_Guide_EN_NOV24_0053016.pdf'),
        'plan_type': 'IHHP', 'plan_tier': 'IHHP',
        'plan_display': 'Bupa IHHP', 'region': 'Global',
        'is_modular': True, 'source_id': 'bupa_ihhp',
        'is_spread': False
    },
    {
        'path': os.path.join(PDF_BASE_DIR, 'Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf'),
        'plan_type': 'GHP', 'plan_tier': 'Select',
        'plan_display': 'Bupa GHP Select', 'region': 'Global',
        'is_modular': False, 'source_id': 'bupa_ghp_select',
        'is_spread': True
    },
    {
        'path': os.path.join(PDF_BASE_DIR, 'Bupa-Global-DAC-Major-Medical-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf'),
        'plan_type': 'GHP', 'plan_tier': 'MajorMedical',
        'plan_display': 'Bupa GHP Major Medical', 'region': 'Global',
        'is_modular': False, 'source_id': 'bupa_ghp_major_medical',
        'is_spread': True
    },
    {
        'path': os.path.join(PDF_BASE_DIR, 'Bupa-Global-DAC-Premier-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf'),
        'plan_type': 'GHP', 'plan_tier': 'Premier',
        'plan_display': 'Bupa GHP Premier', 'region': 'Global',
        'is_modular': False, 'source_id': 'bupa_ghp_premier',
        'is_spread': True
    },
    {
        'path': os.path.join(PDF_BASE_DIR, 'UAE-Elite-Global-Health-Plan-Membership-Guide-EN-NOV24-0051587.pdf'),
        'plan_type': 'GHP', 'plan_tier': 'Elite',
        'plan_display': 'Bupa UAE Elite', 'region': 'UAE',
        'is_modular': False, 'source_id': 'bupa_ghp_elite_uae',
        'is_spread': True
    },
    {
        'path': os.path.join(PDF_BASE_DIR, 'DAC_Ultimate_Global_Health_Plan_Membership_Guide_EN_DEC24_0054831.pdf'),
        'plan_type': 'GHP', 'plan_tier': 'Ultimate',
        'plan_display': 'Bupa Ultimate', 'region': 'Global',
        'is_modular': False, 'source_id': 'bupa_ghp_ultimate',
        'is_spread': True
    },
]

print('📂 PDF 파일 확인:')
for cfg in PDF_CONFIGS:
    exists = os.path.exists(cfg['path'])
    print(f"  {'✅' if exists else '❌'} [{cfg['plan_tier']:12s}] {os.path.basename(cfg['path'])}")

📂 PDF 파일 확인:
  ✅ [IHHP        ] HKX_IHHP_Membership_Guide_EN_NOV24_0053016.pdf
  ✅ [Select      ] Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf
  ✅ [MajorMedical] Bupa-Global-DAC-Major-Medical-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf
  ✅ [Premier     ] Bupa-Global-DAC-Premier-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf
  ✅ [Elite       ] UAE-Elite-Global-Health-Plan-Membership-Guide-EN-NOV24-0051587.pdf
  ✅ [Ultimate    ] DAC_Ultimate_Global_Health_Plan_Membership_Guide_EN_DEC24_0054831.pdf


## 3. 섹션 감지 엔진

각 페이지의 텍스트에서 헤더 키워드를 탐지해 `section_type`을 결정합니다.  
키워드 우선순위: 먼저 매칭된 섹션이 적용됩니다.

In [5]:
from typing import Optional

# ─────────────────────────────────────────────────────────────────
# 제외 키워드 — 이 키워드가 페이지 상단에 있으면 통째로 버림
# ─────────────────────────────────────────────────────────────────
EXCLUDE_PATTERNS = [
    r'welcome to membersworld',
    r'how to access membersworld',
    r'wellbeing services',
    r'global virtual care',
    r'second medical opinion',   # wellbeing 섹션 내 홍보성
    r'privacy notice',
    r'data protection',
    r'round the clock reassurance',
    r'a guide to your .* health plan',  # 표지
    r'^hello\b',                         # Hello 인트로
    r'bupa global offers you',           # 마지막 연락처 페이지
    r'general services:.*\+44',         # 연락처 페이지
]

# ─────────────────────────────────────────────────────────────────
# 섹션 타입 감지 키워드 (우선순위 순서)
# ─────────────────────────────────────────────────────────────────
SECTION_PATTERNS = [
    ('benefit_table',    [
        r'table of benefits',
        r'benefit and explanation',
        r'overall annual (policy )?maximum',
        r'paid in full',
        # --- 수정한 부분: 키워드를 분리해서 유연하게 감지 ---
        r'hospital plan',    # 'Hospital Plan' 및 'Hospital Plan (continued)' 모두 감지
        r'module \d',        # 'Module 1', 'Module 2' 등 숫자 붙은 모듈 감지
        r'modules 4',        # 'Modules 4A and 4B' 감지
    ]),
    ('exclusion',        [
        r'general exclusions',
        r'what is not covered',
        r'your exclusions',
        r'exceptions to cover',
    ]),
    ('claim_process',    [
        r'the claiming process',
        r'how to (make|submit) a claim',
        r'pay and claim',
        r'direct (payment|settlement)',
        r'claiming process',
    ]),
    ('pre_auth',         [
        r'pre.?authoris',
        r'need treatment',
        r'mandatory pre.?authorisation',
        r'how to pre.?authorise',
        r'our approach to costs',
    ]),
    ('glossary',         [
        r'^glossary\b',
        r'defined terms?\s+description',
        r'defined term\b',
    ]),
    ('membership_admin', [
        r'want to add more people',
        r'adding your (newborn|dependant)',
        r'children covered at no additional cost',
        r'dependants',
        r'newborn (application|care|child)',
    ]),
    ('terms_conditions', [
        r'terms and conditions',
        r'your policy\b',
        r'^\s*\d+\.\s+(your policy|your cover|premium and payment|renewal|changes to your policy|ending this policy)',
    ]),
]


def detect_section_type(page_text: str) -> Optional[str]:
    """
    페이지 텍스트에서 section_type을 감지.
    - None 반환 시 제외 대상
    - 감지 범위: 페이지 첫 200자 (헤더 위치)
    """
    # 소문자 변환 + 앞부분만 검사 (속도 최적화)
    text_lower = page_text.lower().strip()
    header_zone = text_lower[:200]

    # 1. 제외 패턴 먼저 체크
    for pattern in EXCLUDE_PATTERNS:
        if re.search(pattern, header_zone, re.IGNORECASE | re.MULTILINE):
            return None

    # 2. 섹션 타입 감지 (전체 텍스트 대상)
    for section_type, patterns in SECTION_PATTERNS:
        for pattern in patterns:
            if re.search(pattern, text_lower, re.IGNORECASE | re.MULTILINE):
                return section_type

    # 3. 어떤 섹션에도 안 걸리면 None (제외)
    return None


# ─────────────────────────────────────────────────────────────────
# 텍스트 정제 함수
# ─────────────────────────────────────────────────────────────────
def clean_page_text(text: str) -> str:
    """
    페이지 텍스트에서 노이즈 제거:
    - 페이지 번호 (독립 줄의 숫자)
    - URL (www., https://)
    - 과도한 공백/빈 줄
    - 저작권 문구
    """
    # URL 제거 (membersworld, bupaglobal.com 등)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # 독립 페이지 번호 줄 제거 (줄 전체가 숫자만)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)

    # 저작권·면책 문구 제거
    text = re.sub(
        r'Bupa Global is a trade name.*?Blue Cross and Blue Shield Association\.',
        '', text, flags=re.DOTALL | re.IGNORECASE
    )

    # 3개 이상 연속 빈 줄 → 2개로
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


# ─────────────────────────────────────────────────────────────────
# 단위 테스트
# ─────────────────────────────────────────────────────────────────
test_cases = [
    ("TABLE OF BENEFITS\nOVERALL ANNUAL POLICY MAXIMUM\nUnlimited",    'benefit_table'),
    ("GENERAL EXCLUSIONS\nAdministration fees...",                      'exclusion'),
    ("THE CLAIMING PROCESS\nWhether you choose direct payment...",      'claim_process'),
    ("NEED TREATMENT?\nOur approach to costs...",                       'pre_auth'),
    ("GLOSSARY\nActive treatment\nTreatment from a medical...",         'glossary'),
    ("WANT TO ADD MORE PEOPLE TO YOUR HEALTH PLAN?",                    'membership_admin'),
    ("TERMS AND CONDITIONS\n1. Your policy",                            'terms_conditions'),
    ("WELCOME TO MEMBERSWORLD\nHow to access MembersWorld",             None),
    ("WELLBEING SERVICES\nGlobal Virtual Care",                         None),
    ("PRIVACY NOTICE\nLast updated: September 2023",                    None),
    ("A GUIDE TO YOUR ULTIMATE GLOBAL HEALTH PLAN",                     None),
]

print('🔍 섹션 감지 테스트:')
all_pass = True
for text, expected in test_cases:
    result = detect_section_type(text)
    status = '✅' if result == expected else '❌'
    if result != expected:
        all_pass = False
    print(f"  {status} '{text[:45]:<45}' → {result} (expected: {expected})")

print(f"\n{'✅ 모든 테스트 통과' if all_pass else '❌ 일부 테스트 실패 — 패턴 수정 필요'}")

🔍 섹션 감지 테스트:
  ✅ 'TABLE OF BENEFITS
OVERALL ANNUAL POLICY MAXIM' → benefit_table (expected: benefit_table)
  ✅ 'GENERAL EXCLUSIONS
Administration fees...    ' → exclusion (expected: exclusion)
  ✅ 'THE CLAIMING PROCESS
Whether you choose direc' → claim_process (expected: claim_process)
  ✅ 'NEED TREATMENT?
Our approach to costs...     ' → pre_auth (expected: pre_auth)
  ✅ 'GLOSSARY
Active treatment
Treatment from a me' → glossary (expected: glossary)
  ✅ 'WANT TO ADD MORE PEOPLE TO YOUR HEALTH PLAN? ' → membership_admin (expected: membership_admin)
  ✅ 'TERMS AND CONDITIONS
1. Your policy          ' → terms_conditions (expected: terms_conditions)
  ✅ 'WELCOME TO MEMBERSWORLD
How to access Members' → None (expected: None)
  ✅ 'WELLBEING SERVICES
Global Virtual Care       ' → None (expected: None)
  ✅ 'PRIVACY NOTICE
Last updated: September 2023  ' → None (expected: None)
  ✅ 'A GUIDE TO YOUR ULTIMATE GLOBAL HEALTH PLAN  ' → None (expected: None)

✅ 모든 테스트 통과


## 4. 섹션 인식 PDF 로더

PyMuPDF로 페이지 단위 로딩 + 섹션 감지 + 정제를 동시에 수행합니다.

In [6]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
import re

def extract_right_page_number(text: str) -> int | None:
    matches = re.findall(r'(?:^|\n)(\d{1,3})(?=\n|$)', text)
    candidates = [int(m) for m in matches if 1 <= int(m) <= 200]
    if candidates:
        return candidates[-1]
    return None

def load_pdf_with_sections(cfg: dict, verbose: bool = True) -> list[Document]:
    docs = []
    stats = {k: 0 for k in ['benefit_table','exclusion','claim_process',
                              'pre_auth','glossary','membership_admin',
                              'terms_conditions','excluded']}

    doc = fitz.open(cfg['path'])

    for page_num, page in enumerate(doc, start=1):
        page_parts = []

        if cfg.get('is_spread', False):
            if page_num == 1:
                page_parts.append((page.get_text(), "1"))
            else:
                rect = page.rect
                mid_x = rect.width / 2

                left_idx = (page_num - 1) * 2
                right_idx = (page_num - 1) * 2 + 1

                left_rect = fitz.Rect(rect.x0, rect.y0, mid_x, rect.y1)
                page_parts.append((page.get_text(clip=left_rect), str(left_idx)))

                right_rect = fitz.Rect(mid_x, rect.y0, rect.x1, rect.y1)
                page_parts.append((page.get_text(clip=right_rect), str(right_idx)))
        else:
            page_parts.append((page.get_text(), str(page_num)))

        for raw_text, page_label in page_parts:
            if not raw_text.strip():
                continue

            section_type = detect_section_type(raw_text)

            if section_type is None:
                stats['excluded'] += 1
                continue

            clean_text = clean_page_text(raw_text)
            if len(clean_text) < 80:
                stats['excluded'] += 1
                continue

            doc_obj = Document(
                page_content=clean_text,
                metadata={
                    'plan_type':    cfg['plan_type'],
                    'plan_tier':    cfg['plan_tier'],
                    'plan_display': cfg['plan_display'],
                    'region':       cfg['region'],
                    'is_modular':   str(cfg['is_modular']),
                    'source_id':    cfg['source_id'],
                    'section_type': section_type,
                    'chunk_type':   'text',
                    'page':         page_label,
                    'physical_page': page_num,
                    'source':       os.path.basename(cfg['path'])
                }
            )
            docs.append(doc_obj)
            stats[section_type] += 1

    doc.close()

    if verbose:
        included = len(docs)
        print(f"  [{cfg['plan_tier']:12s}] 처리 완료: {included}면 로드됨")

    return docs

# 전체 PDF 로드
print('📄 PDF 로드 및 섹션 분류:\n')
all_raw_docs = []

for cfg in PDF_CONFIGS:
    if not os.path.exists(cfg['path']):
        print(f"  ⚠️  건너뜀: {cfg['plan_tier']} (파일 없음)")
        continue
    docs = load_pdf_with_sections(cfg, verbose=True)
    all_raw_docs.extend(docs)
    print()

print(f'✅ 전체 포함 페이지: {len(all_raw_docs)}개')

from collections import Counter
section_dist = Counter(d.metadata['section_type'] for d in all_raw_docs)
print('\n📊 section_type 분포:')
for stype, count in sorted(section_dist.items(), key=lambda x: -x[1]):
    print(f'  {stype:<22s}: {count}p')

📄 PDF 로드 및 섹션 분류:

  [IHHP        ] 처리 완료: 23면 로드됨

  [Select      ] 처리 완료: 31면 로드됨

  [MajorMedical] 처리 완료: 31면 로드됨

  [Premier     ] 처리 완료: 34면 로드됨

  [Elite       ] 처리 완료: 38면 로드됨

  [Ultimate    ] 처리 완료: 39면 로드됨

✅ 전체 포함 페이지: 196개

📊 section_type 분포:
  benefit_table         : 104p
  pre_auth              : 34p
  membership_admin      : 30p
  terms_conditions      : 10p
  glossary              : 7p
  exclusion             : 6p
  claim_process         : 5p


### 표 추출

In [7]:
import pdfplumber
from langchain_core.documents import Document
import os

def table_to_text(table: list, is_modular: bool = False) -> str:
    rows = []
    symbol_map = {
        '✔': 'Covered', '✓': 'Covered', '●': 'Covered', 'O': 'Covered',
        '\uf0fc': 'Covered', '\u2713': 'Covered',
        '✘': 'Not Covered', 'X': 'Not Covered',
        'N/A': 'Not Covered'
        # '-' 제거 → 빈 셀만 Not Covered 처리
    }

    header_row = None

    for row_idx, row in enumerate(table):
        cleaned_row = []
        for c in row:
            if c is not None and str(c).strip():
                text = str(c).strip().replace('\n', ' ')
                for symbol, meaning in symbol_map.items():
                    if text == symbol:
                        text = meaning
                cleaned_row.append(text)
            else:
                cleaned_row.append('')  # 빈 셀은 빈 문자열로 (Not Covered 남발 방지)

        # 헤더 행 저장 (첫 번째 행)
        if row_idx == 0:
            header_row = cleaned_row

        # IHHP 모듈형: 행에 Covered/Not Covered 혼합이면 태그 추가
        if is_modular and header_row:
            has_covered = any(
                'Covered' in c and c != 'Not Covered'
                for c in cleaned_row
            )
            has_not_covered = any(c == 'Not Covered' for c in cleaned_row)
            if has_covered and has_not_covered:
                cleaned_row.append('[일부 모듈 가입 시 보장]')

        rows.append(' | '.join(c for c in cleaned_row if c))  # 빈 셀 제외하고 join
    return '\n'.join(rows)

def is_meaningful_table(table: list) -> bool:
    if not table or len(table) < 2:
        return False
    non_empty = sum(1 for row in table for c in row if c and str(c).strip())
    return non_empty >= 4

all_table_docs = []

for cfg in PDF_CONFIGS:
    if not os.path.exists(cfg['path']):
        continue

    count = 0
    current_section = None
    is_modular = cfg['is_modular']  # bool 그대로 사용

    with pdfplumber.open(cfg['path']) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):

            sides = []
            if cfg.get('is_spread', False):
                if page_num == 1:
                    sides.append((page, "1"))
                else:
                    width = page.width
                    height = page.height
                    left_idx = (page_num - 1) * 2
                    right_idx = (page_num - 1) * 2 + 1
                    left_side = page.crop((0, 0, width/2, height))
                    right_side = page.crop((width/2, 0, width, height))
                    sides.append((left_side, str(left_idx)))
                    sides.append((right_side, str(right_idx)))
            else:
                sides.append((page, str(page_num)))

            for side_page, page_label in sides:
                page_text = side_page.extract_text() or ''

                detected_section = detect_section_type(page_text)
                if detected_section is not None:
                    current_section = detected_section

                if current_section not in ('benefit_table', 'exclusion', 'pre_auth'):
                    continue

                custom_settings = {
                    "vertical_strategy": "text",
                    "horizontal_strategy": "text",
                    "snap_tolerance": 3,
                }

                extracted_tables = side_page.extract_tables(table_settings=custom_settings)

                for tbl_idx, table in enumerate(extracted_tables):
                    if not is_meaningful_table(table):
                        continue

                    table_text = table_to_text(table, is_modular=is_modular)
                    if len(table_text) < 50:
                        continue

                    modular_note = (
                        "\n※ 모듈형 표: Not Covered는 해당 모듈 미가입 상태."
                        " 모듈 가입 시 보장 항목은 [일부 모듈 가입 시 보장] 태그로 표시됩니다.\n"
                        if is_modular else "\n"
                    )

                    doc = Document(
                        page_content=(
                            f"[{cfg['plan_display']} | {current_section} Table | p.{page_label}]"
                            f"{modular_note}"
                            f"{table_text}"
                        ),
                        metadata={
                            'plan_type':     cfg['plan_type'],
                            'plan_tier':     cfg['plan_tier'],
                            'plan_display':  cfg['plan_display'],
                            'region':        cfg['region'],
                            'is_modular':    str(is_modular),
                            'source_id':     cfg['source_id'],
                            'section_type':  current_section,
                            'chunk_type':    'table',
                            'page':          page_label,
                            'table_idx':     tbl_idx,
                            'physical_page': page_num,
                            'source':        os.path.basename(cfg['path'])
                        }
                    )
                    all_table_docs.append(doc)
                    count += 1

    print(f"  📊 [{cfg['plan_tier']:12s}] 유효 표 {count}개 추출 완료")

print(f"\n✅ 총 테이블 청크: {len(all_table_docs)}개")

  📊 [IHHP        ] 유효 표 22개 추출 완료
  📊 [Select      ] 유효 표 18개 추출 완료
  📊 [MajorMedical] 유효 표 18개 추출 완료
  📊 [Premier     ] 유효 표 20개 추출 완료
  📊 [Elite       ] 유효 표 25개 추출 완료
  📊 [Ultimate    ] 유효 표 24개 추출 완료

✅ 총 테이블 청크: 127개


## 5. 섹션별 차등 청크 분할

섹션 특성에 맞게 청크 크기를 다르게 설정합니다.

| section_type | chunk_size | overlap | 이유 |
|---|---|---|---|
| benefit_table | 400 | 60 | 표는 짧고 독립적인 항목들 |
| exclusion | 500 | 80 | 제외 이유 설명 문장 포함 |
| glossary | 300 | 40 | 용어 정의는 짧고 독립적 |
| 나머지 | 600 | 80 | 절차 설명은 문맥 연속성 중요 |

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from collections import Counter

# ─────────────────────────────────────────────────────────────────
# [수정 포인트] BGE-M3 및 보험 약관 맥락 보존을 위한 설정 최적화
# - chunk_size: 400~600 -> 1000~1200 (맥락 보존)
# - chunk_overlap: 60~80 -> 150~200 (문장 끊김 방지)
# ─────────────────────────────────────────────────────────────────
# CHUNK_CONFIGS 원복
CHUNK_CONFIGS = {
    'benefit_table':    {'chunk_size': 1000, 'chunk_overlap': 150},
    'exclusion':        {'chunk_size': 1200, 'chunk_overlap': 200},
    'glossary':         {'chunk_size': 800,  'chunk_overlap': 100},
    'claim_process':    {'chunk_size': 1000, 'chunk_overlap': 150},
    'pre_auth':         {'chunk_size': 1000, 'chunk_overlap': 150},
    'waiting_period':   {'chunk_size': 800,  'chunk_overlap': 100},
    'membership_admin': {'chunk_size': 1000, 'chunk_overlap': 150},
    'terms_conditions': {'chunk_size': 1200, 'chunk_overlap': 200},
}
DEFAULT_CHUNK_CONFIG = {'chunk_size': 1000, 'chunk_overlap': 150}

# COMMON_SEPARATORS 원복
COMMON_SEPARATORS = ['\n\n', '\n', '. ', ' ', '']


def split_docs_by_section(docs: list[Document]) -> list[Document]:
    """섹션 타입별로 최적화된 청크 크기를 적용해 분할"""
    # 섹션 타입별로 그룹화
    groups: dict[str, list[Document]] = {}
    for doc in docs:
        stype = doc.metadata.get('section_type', 'unknown')
        groups.setdefault(stype, []).append(doc)

    all_chunks = []
    for stype, group_docs in groups.items():
        cfg = CHUNK_CONFIGS.get(stype, DEFAULT_CHUNK_CONFIG)
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg['chunk_size'],
            chunk_overlap=cfg['chunk_overlap'],
            separators=COMMON_SEPARATORS,
            length_function=len # 문자열 길이 기준
        )
        
        chunks = splitter.split_documents(group_docs)
        all_chunks.extend(chunks)

    return all_chunks

# 실행 및 결과 확인
all_text_chunks = split_docs_by_section(all_raw_docs)

print(f'✅ 총 텍스트 청크: {len(all_text_chunks)}개')
print('\n📊 섹션별 청크 수:')
chunk_dist = Counter(d.metadata['section_type'] for d in all_text_chunks)
for stype, count in sorted(chunk_dist.items(), key=lambda x: -x[1]):
    cfg = CHUNK_CONFIGS.get(stype, DEFAULT_CHUNK_CONFIG)
    print(f'  {stype:<22s}: {count:3d}청크  (chunk_size={cfg["chunk_size"]}, overlap={cfg["chunk_overlap"]})')

c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 총 텍스트 청크: 737개

📊 섹션별 청크 수:
  benefit_table         : 375청크  (chunk_size=1000, overlap=150)
  pre_auth              : 134청크  (chunk_size=1000, overlap=150)
  membership_admin      : 123청크  (chunk_size=1000, overlap=150)
  glossary              :  36청크  (chunk_size=800, overlap=100)
  terms_conditions      :  32청크  (chunk_size=1200, overlap=200)
  exclusion             :  22청크  (chunk_size=1200, overlap=200)
  claim_process         :  15청크  (chunk_size=1000, overlap=150)


## 7. 전처리 결과 검증

실제 청크 샘플을 확인해 전처리 품질을 검증합니다.

In [9]:
import random
from collections import Counter

# 1. 통합 데이터 생성
all_docs = all_text_chunks + all_table_docs

print('=' * 70)
print(f'📦 [최종 데이터 통계]')
print(f'   - 전체 청크 수: {len(all_docs)}개')
print(f'   - 텍스트 청크: {len(all_text_chunks)}개')
print(f'   - 테이블 청크: {len(all_table_docs)}개')
print('=' * 70)

# 2. 플랜별 청크 분포
print('\n📊 플랜별 데이터 분포:')
plan_dist = Counter(d.metadata['plan_tier'] for d in all_docs)
for tier, count in plan_dist.most_common():
    print(f'   {tier:<15s}: {count:4d}개')

# 3. 섹션 타입별 청크 분포
print('\n📊 섹션(Section)별 데이터 분포:')
final_dist = Counter(d.metadata['section_type'] for d in all_docs)
for stype, count in final_dist.most_common():
    print(f'   {stype:<22s}: {count:4d}개')

# 4. 스프레드 분할 검증
print('\n🧐 스프레드 분할 검증 (Physical Page 4 -> Logical Page 6, 7 확인):')
sample_tier = all_docs[0].metadata['plan_tier']
spread_samples = [d for d in all_docs
                  if d.metadata.get('physical_page') == 4
                  and d.metadata['plan_tier'] == sample_tier]
pages_found = sorted(list(set(d.metadata['page'] for d in spread_samples)))
print(f'   [{sample_tier}] 물리 4페이지 내 발견된 논리 페이지: {pages_found}')

# 5. 랜덤 샘플 검수
print('\n🔍 섹션별 랜덤 샘플 상세 확인 (내용 및 메타데이터):')
print('-' * 70)

for stype in sorted(final_dist.keys()):
    samples = [d for d in all_docs if d.metadata['section_type'] == stype]
    if samples:
        s = random.choice(samples)
        m = s.metadata

        print(f"[{stype.upper()}] | {m['plan_display']} | p.{m['page']} (물리 p.{m.get('physical_page')})")
        print(f"타입: {m['chunk_type']} | 소스: {m['source']}")

        content_preview = s.page_content[:250].replace('\n', ' ↵ ')
        print(f"내용: {content_preview}...")
        print('-' * 70)

📦 [최종 데이터 통계]
   - 전체 청크 수: 864개
   - 텍스트 청크: 737개
   - 테이블 청크: 127개

📊 플랜별 데이터 분포:
   Elite          :  178개
   Ultimate       :  170개
   Premier        :  138개
   MajorMedical   :  130개
   Select         :  127개
   IHHP           :  121개

📊 섹션(Section)별 데이터 분포:
   benefit_table         :  465개
   pre_auth              :  160개
   membership_admin      :  123개
   glossary              :   36개
   exclusion             :   33개
   terms_conditions      :   32개
   claim_process         :   15개

🧐 스프레드 분할 검증 (Physical Page 4 -> Logical Page 6, 7 확인):
   [IHHP] 물리 4페이지 내 발견된 논리 페이지: ['4']

🔍 섹션별 랜덤 샘플 상세 확인 (내용 및 메타데이터):
----------------------------------------------------------------------
[BENEFIT_TABLE] | Bupa UAE Elite | p.26 (물리 p.14)
타입: text | 소스: UAE-Elite-Global-Health-Plan-Membership-Guide-EN-NOV24-0051587.pdf
내용: HEARING AND VISION AIDS, AND VISION CORRECTION BY SURGERIES AND LASER ↵ We pay for hearing and vision aids, and vision correction by surgeries and laser in the case ↵ of 

## 8. 임베딩 + Chroma 저장

bge-m3 다국어 임베딩으로 벡터화합니다.

In [10]:
import os
import torch
import gc
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CHROMA_DIR = './chroma_bupa_preprocessed'

# 기존 DB 초기화 (LlamaParse로 재인덱싱이므로 기존 데이터 삭제)
if os.path.exists(CHROMA_DIR):
    import shutil
    shutil.rmtree(CHROMA_DIR)
    print('🗑️  기존 ChromaDB 삭제 완료')

print(f'⏳ 모델 로드 중 ({device} / fp16 모드)...')
model_kwargs = {
    'device': device,
    'model_kwargs': {'torch_dtype': torch.float16}
}
encode_kwargs = {'normalize_embeddings': True, 'batch_size': 8}

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

if hasattr(embedding_model, 'client'):
    embedding_model.client.max_seq_length = 768
elif hasattr(embedding_model, 'model'):
    embedding_model.model.max_seq_length = 768
else:
    embedding_model.__dict__['_client'].max_seq_length = 768

# ID 생성 (all_docs = all_text_chunks + all_table_docs)
doc_ids = [
    f"{d.metadata.get('plan_tier')}_p{d.metadata.get('page')}_{d.metadata.get('chunk_type')}_{i}"
    for i, d in enumerate(all_docs)
]

vectorstore = Chroma(
    collection_name='bupa_preprocessed',
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR
)

print(f'⏳ 벡터 저장 시작 (총 {len(all_docs)}개)...')
safe_batch_size = 8

for i in range(0, len(all_docs), safe_batch_size):
    batch_docs = all_docs[i : i + safe_batch_size]
    batch_ids  = doc_ids[i : i + safe_batch_size]

    try:
        with torch.inference_mode():
            vectorstore.add_documents(documents=batch_docs, ids=batch_ids)
    except torch.cuda.OutOfMemoryError:
        print("⚠️ OOM 감지! 메모리 비우기 수행 중...")
        torch.cuda.empty_cache()
        gc.collect()

    if (i // safe_batch_size) % 5 == 0:
        print(f"📦 [{i + len(batch_docs)}/{len(all_docs)}] 완료 (VRAM 사용중)")

    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

print(f'\n✅ 성공적으로 저장되었습니다! 총 {len(all_docs)}개 청크.')
print(f'📊 최종 벡터 수: {vectorstore._collection.count()}개')

🗑️  기존 ChromaDB 삭제 완료
⏳ 모델 로드 중 (cuda / fp16 모드)...


Loading weights: 100%|██████████| 391/391 [00:03<00:00, 106.01it/s]


⏳ 벡터 저장 시작 (총 864개)...
📦 [8/864] 완료 (VRAM 사용중)
📦 [48/864] 완료 (VRAM 사용중)
📦 [88/864] 완료 (VRAM 사용중)
📦 [128/864] 완료 (VRAM 사용중)
📦 [168/864] 완료 (VRAM 사용중)
📦 [208/864] 완료 (VRAM 사용중)
📦 [248/864] 완료 (VRAM 사용중)
📦 [288/864] 완료 (VRAM 사용중)
📦 [328/864] 완료 (VRAM 사용중)
📦 [368/864] 완료 (VRAM 사용중)
📦 [408/864] 완료 (VRAM 사용중)
📦 [448/864] 완료 (VRAM 사용중)
📦 [488/864] 완료 (VRAM 사용중)
📦 [528/864] 완료 (VRAM 사용중)
📦 [568/864] 완료 (VRAM 사용중)
📦 [608/864] 완료 (VRAM 사용중)
📦 [648/864] 완료 (VRAM 사용중)
📦 [688/864] 완료 (VRAM 사용중)
📦 [728/864] 완료 (VRAM 사용중)
📦 [768/864] 완료 (VRAM 사용중)
📦 [808/864] 완료 (VRAM 사용중)
📦 [848/864] 완료 (VRAM 사용중)

✅ 성공적으로 저장되었습니다! 총 864개 청크.
📊 최종 벡터 수: 864개


---
## 📝 이 노트북을 LangGraph 챗봇에 연결하는 방법

LangGraph 노트북(`bupa_rag_langgraph.ipynb`)에서 다음 두 곳을 수정하세요.

### 1) vectorstore 로드 → 이 노트북의 DB 사용
```python
CHROMA_DIR = './chroma_bupa_preprocessed'
vectorstore = Chroma(
    collection_name='bupa_preprocessed',
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR
)
```
### 4) generate_node에서 language 활용
```python
def generate_node(state: ChatState) -> dict:
    normalized = state.get('normalized', {})
    language = normalized.get('language', 'ko')
    # SYSTEM_PROMPT에서 answer_language를 동적으로 삽입
    ...
```